# RMT A4 — test-set evaluation of the replication runs (Qwen3-0.6B-Base, MLC, seed 1)
Self-contained: clones the project repo and adds the evaluation code inline (incl. the adapter fp16 fix and the 256-token generation cap with batch 16, deviations 12 and 14).
Needs: **GPU T4 x2**, **Internet on**. Checkpoints: Rakhat3110/rmt-peft-ckpts (public; if private, attach the HF_TOKEN secret).
Evaluates Stage 0 + the end of every stage for full, LoRA and adapter + the LoRA matched checkpoint = 11 checkpoints on 2 GPUs (~5-6 h). Output: `eval_results.zip`.


In [ ]:
import os, subprocess, sys, time, glob, json, base64, io, tarfile
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_ALLOW_CODE_EVAL"] = "1"
os.environ["CKPT_CACHE"] = "/tmp/ckpt_cache"
HUB_REPO  = "Rakhat3110/rmt-peft-ckpts"
PLAN_ARGS = "--model replication --methods full,lora,adapter --seeds 1 --stage0 --name participant2"
N_GPUS = 2
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret attached")
except Exception:
    print("no HF_TOKEN secret - using public access")
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/alishermede/parameter-efficient-fine-tuning-and-catastrophic-forgetting-in-small-language-models.git repo
%cd repo
!git log -1 --format="repo commit %H  %s"
PATCH_B64 = "..."
import hashlib
assert hashlib.sha256(PATCH_B64.encode()).hexdigest() == "9c8efeae1ccf15dcbb6a3dfeaa15e693b2b53044b5eaad677137c0c81a03313e", "embedded code corrupted"
tarfile.open(fileobj=io.BytesIO(base64.b64decode(PATCH_B64)), mode="r:xz").extractall(".")
!ls src tasks configs
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao

In [ ]:
# rebuild data/test/ and verify hashes (must print 'verified')
!python src/prepare_data.py

In [ ]:
# PILOT: Stage 0, 50 items per task (never reported). Stops the notebook if scores look broken.
!python src/eval.py --run_id q3-06b_full_MLC_s1 --checkpoint stage0 --pilot 50 --hub_repo {HUB_REPO} --strict_gen_cap --gen_batch_size 16 --overwrite
s = json.load(open("results/eval_pilot/q3-06b_base/stage0/summary.json"))["scores"]
print({k: round(100*v["score"], 1) for k, v in s.items()})
if s["mmlu_primary"]["score"] < 0.32 or s["gsm8k"]["score"] < 0.10:
    raise SystemExit("Stage-0 pilot looks broken (near chance) - stop and check fp16 / prompts")

In [ ]:
# FULL evaluation: one worker per GPU
!python src/plan_eval.py {PLAN_ARGS} --gpus {N_GPUS}
open("results/eval/jobs/participant2_gpu1.txt", "a").write("q3-06b_lora_MLC_s1 stage1_medmcqa_050\n")  # matched checkpoint (T = 47.5%)
files = sorted(glob.glob("results/eval/jobs/participant2_gpu*.txt"))
procs = []
for gpu, jf in enumerate(files):
    log = open(f"/kaggle/working/eval_gpu{gpu}.log", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    procs.append(subprocess.Popen([sys.executable, "src/eval.py", "--jobs", jf, "--hub_repo", HUB_REPO, "--strict_gen_cap", "--gen_batch_size", "16", "--keep_going"],
                                  env=env, stdout=log, stderr=subprocess.STDOUT))
    print("GPU", gpu, open(jf).read().count("\n"), "jobs")
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(300)
    print(f"{(time.time()-t0)/60:5.0f} min  done checkpoints: {len(glob.glob('results/eval/*/*/summary.json'))}", flush=True)
print("exit codes:", [p.returncode for p in procs])

In [ ]:
for f in sorted(glob.glob("results/eval/*/*/summary.json")):
    s = json.load(open(f)); sc = s.get("scores", {})
    print(f"{s.get('run_id','?'):24s} {s.get('checkpoint','?'):20s} " + "  ".join(f"{k}={100*v['score']:.1f}" for k, v in sc.items()))
!grep -h -E "FAILED|Traceback|all jobs done" /kaggle/working/eval_gpu*.log | head -20

In [ ]:
!cp /kaggle/working/eval_gpu*.log results/eval/ 2>/dev/null; cd /kaggle/working && rm -f eval_results.zip && zip -qr eval_results.zip repo/results/eval repo/results/eval_pilot repo/results/tables && ls -la eval_results.zip